# Tiny Recursive Models with Drake and Panda

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, Video, display

import libero
import trm

## Experiment settings

In [ ]:
DATA_DIR = Path("data/LIBERO/libero_spatial")
ASSET_DIR = Path("external/LIBERO")
RUN_DIR = Path("runs/project")

SEED = 42
DEVICE = "auto"
TASK_IDS = tuple(range(10))
HUMAN_EPISODES = 1
POLICY_EPISODES = 5
MAX_STEPS = 600
SAVE_VIDEO = True

config = trm.TrainingConfig(
    epochs=30,
    batch_size=512,
    save_path=str(RUN_DIR / "model.pt"),
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
trm.seed_everything(SEED)
config

## Download

In [ ]:
if not ASSET_DIR.exists() or not all(
    (DATA_DIR / f"{name}_demo.hdf5").is_file() for name in libero.NAMES
):
    libero.download(data_dir=DATA_DIR, libero_root=ASSET_DIR)

## Human replay

In [ ]:
human_results = libero.replay(
    data_dir=DATA_DIR,
    libero_root=ASSET_DIR,
    task_ids=TASK_IDS,
    episodes=HUMAN_EPISODES,
    output=RUN_DIR / "human",
    video=SAVE_VIDEO,
)

## Training

In [ ]:
trm.seed_everything(SEED)
best_val_loss = trm.train(config, data_dir=DATA_DIR, device=DEVICE)

In [ ]:
history = json.loads(Path(config.save_path).with_suffix(".history.json").read_text())
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([row["epoch"] for row in history], [row["train_loss"] for row in history], label="Training")
ax.plot([row["epoch"] for row in history], [row["val_loss"] for row in history], label="Validation")
ax.set(xlabel="Epoch", ylabel="Loss")
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "training.png", dpi=160)
plt.show()

## Learned-policy evaluation

In [ ]:
policy_results = trm.evaluate(
    checkpoint=config.save_path,
    data_dir=DATA_DIR,
    libero_root=ASSET_DIR,
    task_ids=TASK_IDS,
    episodes=POLICY_EPISODES,
    max_steps=MAX_STEPS,
    output=RUN_DIR / "policy",
    video=SAVE_VIDEO,
    device=DEVICE,
    seed=SEED,
)

In [ ]:
def success_by_task(report):
    return [
        np.mean([row["success"] for row in report["episodes"] if row["task_id"] == task])
        for task in TASK_IDS
    ]


x = np.arange(len(TASK_IDS))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - 0.2, success_by_task(human_results), width=0.4, label="Human replay")
ax.bar(x + 0.2, success_by_task(policy_results), width=0.4, label="TRM policy")
ax.set(xticks=x, xticklabels=TASK_IDS, xlabel="Task ID", ylabel="Success rate", ylim=(0, 1.05))
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "success.png", dpi=160)
plt.show()

## Videos

In [ ]:
if SAVE_VIDEO:
    for label, folder, report in (
        ("Human replay", "human", human_results),
        ("TRM policy", "policy", policy_results),
    ):
        first = report["episodes"][0]
        video = RUN_DIR / folder / f"task_{first['task_id']:02d}_{first['episode']:02d}.mp4"
        display(Markdown(f"**{label}** — {video.name}"))
        display(Video(filename=str(video), embed=True, width=640))

## Conclusions

In [ ]:
display(
    Markdown(
        "| Experiment result | Value |\n|---|---:|\n"
        f"| Human replay success rate | {human_results['success_rate']:.1%} |\n"
        f"| TRM policy success rate | {policy_results['success_rate']:.1%} |\n"
        f"| Best validation loss | {best_val_loss:.5f} |"
    )
)